## Project Framework (CRISP-DM)

### 1. Business Understanding
The goal of this project is to build a fast, automated system that classifies clients' banking requests by intent, minimizing the time it takes to route each request and give the client a correct answer.
- Faster routing reduces manual triage work for support teams.
- Accuracy matters: a misrouted request such as `compromised_card` delays a security issue, not just a reply.

### 2. Data Understanding
- 10,003 training and 3,080 test customer queries, 2 columns (`text`, `category`), 77 intent classes.
- The test set is balanced (40 queries per class); train class balance will be examined in EDA.
- No missing or empty values, no duplicate requests within either set.
- 6 near-duplicate queries shared between train and test (~0.2% of test), left as is due to negligible impact.
- Two inconsistent label names: `reverted_card_payment?` and `Refund_not_showing_up`.

### 3. Data Preparation
- Encode all 77 intent categories as integers.
- **MLP:** convert each query into a numeric vector using TF-IDF features.
- **RoBERTa:** tokenize queries into sub-word tokens, with padding and truncation to give every input the same length.

### 4. Modeling
- **Baseline:** Multi-Layer Perceptron on TF-IDF features. Fast and low-resource.
- **Advanced:** pre-trained RoBERTa transformer, fine-tuned with LoRA to adapt it to banking queries while training only a small fraction of its parameters.

### 5. Evaluation
- **Primary metric: Macro F1:** the average of per-class F1 scores, so all 77 intents count equally and weak performance on rarer intents can't hide behind common ones.
- **Secondary metrics:** accuracy, precision, recall.
- **Per-class comparison:** which intents improve most with RoBERTa, and which remain confused.
- **Efficiency:** training time and number of trainable parameters, to judge whether the extra complexity is justified.

### 6. Deployment
- Save the fine-tuned LoRA-RoBERTa model so it can be reused without retraining.
- Protect clients' sensitive data by redacting PII (card numbers, names, emails, phone numbers) before prediction.
- **Inference pipeline:** customer query → redact PII → predict intent → route to the right team.
- **Monitoring:** track live performance and data drift, and collect new labeled queries for retraining.

# Load and Verify Data

In [23]:
import pandas as pd

df_train = pd.read_csv("datasets/banking77_train.csv")
df_test = pd.read_csv("datasets/banking77_test.csv")

def inspect_dataset(df, name):
    print(f"--- {name} ---")
    display(df.head())
    df.info()
    print(f"Number of missing values:\n{df.isna().sum()}")
    print(f"Number of empty texts: {df['text'].str.strip().eq('').sum()}")
    print(f"Number of duplicated texts: {df['text'].duplicated().sum()}")
    print(f"Number of unique categories: {df['category'].nunique()}")
    

inspect_dataset(df_train, "Train Dataset")
inspect_dataset(df_test, "Test Dataset")

print(f"--- Joint Tests ---")
print(f"Test labels unseen in train: {set(df_test['category']).difference(set(df_train['category']))}")
print(f"Number of shared texts: {len(set(df_train['text'].str.lower().str.strip()).intersection(set(df_test['text'].str.lower().str.strip())))}")

--- Train Dataset ---


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10003 entries, 0 to 10002
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   text      10003 non-null  object
 1   category  10003 non-null  object
dtypes: object(2)
memory usage: 156.4+ KB
Number of missing values:
text        0
category    0
dtype: int64
Number of empty texts: 0
Number of duplicated texts: 0
Number of unique categories: 77
--- Test Dataset ---


,text,category
0,How do I locate my card?,card_arrival
1,"I still have not received my new card, I order...",card_arrival
2,I ordered a card but it has not arrived. Help ...,card_arrival
3,Is there a way to know when my card will arrive?,card_arrival
4,My card has not arrived yet.,card_arrival


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3080 entries, 0 to 3079
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   text      3080 non-null   object
 1   category  3080 non-null   object
dtypes: object(2)
memory usage: 48.3+ KB
Number of missing values:
text        0
category    0
dtype: int64
Number of empty texts: 0
Number of duplicated texts: 0
Number of unique categories: 77
--- Joint Tests ---
Test labels unseen in train: set()
Number of shared texts: 6


### Data Quality Summary

| Check | Train | Test |
|---|---|---|
| Rows | 10,003 | 3,080 |
| Columns | `text`, `category` | `text`, `category` |
| Missing values | 0 | 0 |
| Empty / whitespace-only texts | 0 | 0 |
| Duplicate texts | 0 | 0 |
| Unique categories | 77 | 77 |

**Train vs. test:**
- All test categories are present in train, so there are no unseen labels.
- **0 exact** text overlaps; **6 near-duplicates** after lowercasing and stripping whitespace (~0.2% of test). Left untouched: the impact on evaluation is negligible, and the test set is kept unchanged as the benchmark.

**Observations:**
- Two labels have inconsistent naming: `reverted_card_payment?` (trailing `?`) and `Refund_not_showing_up` (capitalized). They are consistent across train and test, so they don't affect training, but they may be cleaned for readability in plots.
- The test set has exactly 40 queries per class (3,080 = 77 × 40). Train class balance will be examined in EDA.

**Conclusion:** both datasets load correctly and are clean, so they are ready for EDA.